In [7]:
%pip install geopandas fiona pyogrio

import fiona
import geopandas as gpd
import pandas as pd

# Fiona disables KML by default; GDAL's KML driver is available here.
fiona.drvsupport.supported_drivers["KML"] = "r"

# Replace with your downloaded KML path
kml_path = "/Users/priyankaramachandran/Desktop/hackharvest/notebooks/dataset/dairy_farms.kml"

kml_layers = fiona.listlayers(kml_path)
if not kml_layers:
    raise ValueError(f"No KML data layers found in {kml_path}")

dairy_layers = [
    gpd.read_file(kml_path, driver="KML", layer=layer).assign(kml_layer=layer)
    for layer in kml_layers
]
dairy = gpd.GeoDataFrame(
    pd.concat(dairy_layers, ignore_index=True),
    geometry="geometry",
    crs=dairy_layers[0].crs,
)

print("Number of facilities:", len(dairy))
print("Columns:", dairy.columns.tolist())
display(dairy.head())

Defaulting to user installation because normal site-packages is not writeable
DEPRECATION: Loading egg at /Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/seaborn-0.11.2-py3.11.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330

[notice] A new release of pip is available: 24.3.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Number of facilities: 472
Columns: ['Name', 'Description', 'geometry', 'kml_layer']


,Name,Description,geometry,kml_layer
0,LEWIS DAIRY,"<html xmlns:fo=""http://www.w3.org/1999/XSL/For...",POINT Z (-85.32322 31.17398 0.00000),ASHFORD
1,LEWIS FAMILY DAIRY,"<html xmlns:fo=""http://www.w3.org/1999/XSL/For...",POINT Z (-85.32322 31.17398 0.00000),ASHFORD
2,EVANS DAIRY,"<html xmlns:fo=""http://www.w3.org/1999/XSL/For...",POINT Z (-85.61611 34.86361 0.00000),BRYANT
3,PAYTON FARMS,"<html xmlns:fo=""http://www.w3.org/1999/XSL/For...",POINT Z (-85.84227 34.32865 0.00000),COLLINSVILLE
4,CEDARCREST FARMS,"<html xmlns:fo=""http://www.w3.org/1999/XSL/For...",POINT Z (-87.66260 32.42464 0.00000),FAUNSDALE


In [8]:
dairy = dairy.to_crs("EPSG:4326")

# Keep only point geometries
dairy = dairy[dairy.geometry.geom_type == "Point"].copy()

dairy["longitude"] = dairy.geometry.x
dairy["latitude"] = dairy.geometry.y

print("Number of dairy locations:", len(dairy))

display(dairy.head())

Number of dairy locations: 472


,Name,Description,geometry,kml_layer,longitude,latitude
0,LEWIS DAIRY,"<html xmlns:fo=""http://www.w3.org/1999/XSL/For...",POINT Z (-85.32322 31.17398 0.00000),ASHFORD,-85.323220,31.173980
1,LEWIS FAMILY DAIRY,"<html xmlns:fo=""http://www.w3.org/1999/XSL/For...",POINT Z (-85.32322 31.17398 0.00000),ASHFORD,-85.323220,31.173980
2,EVANS DAIRY,"<html xmlns:fo=""http://www.w3.org/1999/XSL/For...",POINT Z (-85.61611 34.86361 0.00000),BRYANT,-85.616111,34.863611
3,PAYTON FARMS,"<html xmlns:fo=""http://www.w3.org/1999/XSL/For...",POINT Z (-85.84227 34.32865 0.00000),COLLINSVILLE,-85.842270,34.328649
4,CEDARCREST FARMS,"<html xmlns:fo=""http://www.w3.org/1999/XSL/For...",POINT Z (-87.66260 32.42464 0.00000),FAUNSDALE,-87.662597,32.424635


In [9]:
# Preserve other attributes for verification
dairy.drop(columns="geometry").to_csv("usa_dairy_farms.csv", index=False)

print("Saved usa_dairy_farms.csv")

Saved usa_dairy_farms.csv


In [10]:
print(dairy.columns.tolist())

if "state" in dairy.columns:
    display(dairy["state"].value_counts())

['Name', 'Description', 'geometry', 'kml_layer', 'longitude', 'latitude']
